In [ ]:
import os
import sys
from pathlib import Path

def find_gptcast_root(start: Path) -> Path:
    p = start.resolve()
    for _ in range(8):
        if (p / 'gptcast').is_dir():
            return p
        if (p / 'GPTCast' / 'gptcast').is_dir():
            return (p / 'GPTCast').resolve()
        p = p.parent
    return start.resolve()

ROOT = find_gptcast_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('Using ROOT:', ROOT)

from gptcast.models import GPTCast, VAEGANVQ
from gptcast.models.components import GPT, GPTCastConfig
from gptcast.data import Era5LandSwvl1DataModule
from gptcast.utils.plotting import plot_era5land_grid, plot_era5land_panel
from gptcast.utils.converters import swvl1_norm_to_phys
import numpy as np
import random
from datetime import datetime
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
from einops import rearrange

# Reproducibility note:
# - `top_k=1` uses greedy decoding (deterministic)
# - `top_k=None` uses multinomial sampling (stochastic)
try:
    from lightning.pytorch import seed_everything
    seed_everything(42, workers=True)
except Exception:
    random.seed(42)
    np.random.seed(42)
    torch.manual_seed(42)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(42)


## Load pretrained model and a soil moisture (swvl1) sequence of data from the dataset


In [ ]:
# ROOT is set in the first cell (auto-detected)

# Default checkpoints copied from the server.
# This notebook uses local checkpoints only.

def classify_ckpt(path: Path) -> str | None:
    ckpt = torch.load(path, map_location='cpu', weights_only=False)
    state_dict = ckpt.get('state_dict', ckpt)
    keys = list(state_dict.keys())
    if any(k.startswith('transformer.') for k in keys):
        return 'gpt'
    if any('quantize.embedding.weight' in k for k in keys):
        return 'first_stage'
    return None


def pick_first_stage_ckpt(root: Path) -> Path:
    preferred = [
        root / 'models/first_stage.ckpt',
        root / 'models/vae.ckpt',
        root / 'models/tokenizer.ckpt',
    ]
    for candidate in preferred:
        if candidate.exists():
            return candidate
    model_dir = root / 'models'
    if model_dir.exists():
        for candidate in sorted(model_dir.glob('*.ckpt')):
            if classify_ckpt(candidate) == 'first_stage':
                return candidate
    raise FileNotFoundError('No local first-stage checkpoint found under models/.')


def pick_gpt_ckpt(root: Path) -> Path:
    preferred = [
        root / 'models/gpt.ckpt',
        root / 'models/forecast.ckpt',
        root / 'models/16x16.ckpt',
    ]
    for candidate in preferred:
        if candidate.exists():
            return candidate
    model_dir = root / 'models'
    if model_dir.exists():
        for candidate in sorted(model_dir.glob('*.ckpt')):
            if classify_ckpt(candidate) == 'gpt':
                return candidate
    raise FileNotFoundError('No local GPT checkpoint found under models/.')

first_stage_ckpt = pick_first_stage_ckpt(ROOT)
gpt_ckpt = pick_gpt_ckpt(ROOT)
gpt_compare_ckpt = None  # e.g. ROOT / 'models/gpt_best.ckpt'

assert first_stage_ckpt.exists(), f"Missing: {first_stage_ckpt}"
assert gpt_ckpt.exists(), f"Missing: {gpt_ckpt}"

def build_transformer() -> GPT:
    return GPT(
        vocab_size=1024,
        block_size=2048,
        n_layer=GPTCastConfig.n_layer,
        n_head=GPTCastConfig.n_head,
        n_embd=GPTCastConfig.n_embd,
    )

def load_gptcast_from_ckpt(gpt_path: Path, first_stage_path: Path) -> GPTCast:
    transformer = build_transformer()
    first_stage = VAEGANVQ.load_from_pretrained(str(first_stage_path), device=device).to(device).eval()
    return GPTCast.load_from_checkpoint(
        str(gpt_path),
        transformer=transformer,
        first_stage=first_stage,
        map_location=device,
    ).to(device).eval()

gpt_main = load_gptcast_from_ckpt(gpt_ckpt, first_stage_ckpt)
gpt_compare = None
if gpt_compare_ckpt is not None:
    gpt_compare_ckpt = Path(gpt_compare_ckpt)
    assert gpt_compare_ckpt.exists(), f'Missing: {gpt_compare_ckpt}'
    gpt_compare = load_gptcast_from_ckpt(gpt_compare_ckpt, first_stage_ckpt)


In [ ]:
# ROOT is set in the first cell (auto-detected)

# The original GPTCast notebook samples a random test sequence.
# For SWVL1 this can be hard to interpret (and can be ocean-dominated). Here we use a
# reproducible example centred over East China, with a small "smart" jitter search to
# keep the crop mostly over land.

import xarray as xr
from datetime import timedelta
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import torch.nn.functional as F

# Example start time (daily at 12:00 in this dataset)
T0 = datetime(2020, 4, 6, 12, 0)
SEQ_LEN = 7

# Data location (yearly NetCDF)
BASE_DIR = ROOT / 'data/0.1/1/land_surface'
NC_NAME = 'volumetric_soil_water_layer_1.nc'

# Training-style preprocessing
CLIP = (0.0, 0.8)
RESIZE = 720
CROP = 256

# East China center (degrees)
ROI_NAME = 'East China'
ROI_CENTER = dict(lat=34.0, lon=116.0)

# Smart-crop around ROI center (on resized grid)
MAX_MASK_FRAC = 0.40
SMART_ATTEMPTS = 50
JITTER_PX = 80

# Local RNG so this ROI crop is reproducible even if other cells also use randomness
rng = random.Random(42)


def open_year_ds(year: int) -> xr.Dataset:
    p = BASE_DIR / str(year) / NC_NAME
    assert p.exists(), f"Missing: {p}"
    return xr.open_dataset(p)


def sel_time_swvl1(ds: xr.Dataset, dt: datetime):
    t = np.datetime64(dt)
    try:
        return ds['swvl1'].sel(time=t)
    except Exception:
        return ds['swvl1'].sel(time=t, method='nearest')


# Load frames (physical units, NaN over ocean)
frames = []
ds_y = open_year_ds(T0.year)
for j in range(SEQ_LEN):
    dt_j = T0 + timedelta(days=j)
    da = sel_time_swvl1(ds_y, dt_j)
    frames.append(da.values.astype(np.float32, copy=False))
frames = np.stack(frames, axis=0)  # (S, lat, lon)

lat_vals = ds_y['latitude'].values
lon_vals = ds_y['longitude'].values

mask_native = np.isnan(frames[0])

# Clip (physical)
cmin, cmax = float(CLIP[0]), float(CLIP[1])
frames = np.nan_to_num(frames, nan=cmin)
frames = np.clip(frames, cmin, cmax)

# Resize to square canvas (paper-style)
x = torch.from_numpy(frames).unsqueeze(1)  # (S,1,H,W)
x = F.interpolate(x, size=(RESIZE, RESIZE), mode='bilinear', align_corners=False)
frames_r = x.squeeze(1).numpy()

m = torch.from_numpy(mask_native.astype(np.float32))[None, None, ...]
m = F.interpolate(m, size=(RESIZE, RESIZE), mode='nearest')
mask_r = m[0, 0].numpy().astype(bool)

# Normalize to [-1,1]
nmin, nmax = -1.0, 1.0
frames_n = (frames_r - cmin) / (cmax - cmin + 1e-12)
frames_n = frames_n * (nmax - nmin) + nmin

# Map ROI center (lat/lon) to resized pixel indices
lat_c = float(ROI_CENTER['lat'])
lon_c = float(ROI_CENTER['lon'])

lat_idx = int(np.argmin(np.abs(lat_vals - lat_c)))
lon_idx = int(np.argmin(np.abs(lon_vals - lon_c)))

y_center = int(round(lat_idx / (len(lat_vals) - 1) * (RESIZE - 1)))
x_center = int(round(lon_idx / (len(lon_vals) - 1) * (RESIZE - 1)))


def clamp(v, lo, hi):
    return int(max(lo, min(hi, v)))


best = None
best_frac = 1.0

for k in range(SMART_ATTEMPTS):
    dy = rng.randint(-JITTER_PX, JITTER_PX) if k > 0 else 0
    dx = rng.randint(-JITTER_PX, JITTER_PX) if k > 0 else 0
    y0 = clamp(y_center - CROP // 2 + dy, 0, RESIZE - CROP)
    x0 = clamp(x_center - CROP // 2 + dx, 0, RESIZE - CROP)

    frac = float(mask_r[y0 : y0 + CROP, x0 : x0 + CROP].mean())
    if frac < best_frac:
        best_frac = frac
        best = (y0, x0)
    if frac <= MAX_MASK_FRAC:
        best = (y0, x0)
        best_frac = frac
        break

if best is None:
    raise RuntimeError('Failed to select a crop')

y0, x0 = best
print(f"ROI {ROI_NAME} center(lat={lat_c}, lon={lon_c}) -> resized center(y={y_center}, x={x_center})")
print(f"Selected crop top-left(y0={y0}, x0={x0}), mask_frac={best_frac:.3f}")

# Global context plot (first frame) + crop box
# Note: this is the resized field (same canvas used for patch extraction).
global0 = np.ma.array(frames_r[0], mask=mask_r)
cm = plt.get_cmap('viridis').copy()
cm.set_bad(color='lightgray')

fig, ax = plt.subplots(1, 1, figsize=(8, 4), dpi=160, layout='constrained')
ax.imshow(global0, cmap=cm, vmin=cmin, vmax=cmax, interpolation='nearest')
ax.add_patch(Rectangle((x0, y0), CROP, CROP, linewidth=1.0, edgecolor='red', facecolor='none'))
ax.set_title(f"Global swvl1 (resized) {T0}  ROI: {ROI_NAME}")
ax.axis('off')
plt.show()

# Build an 'el' dict like the dataloader output (batch_size=1) for downstream cells.
patch = frames_n[:, y0 : y0 + CROP, x0 : x0 + CROP]  # (S,H,W) in [-1,1]
mask_patch = mask_r[y0 : y0 + CROP, x0 : x0 + CROP]

image_hws = np.transpose(patch, (1, 2, 0))  # (H,W,S)
el = {
    'image': torch.from_numpy(image_hws.astype(np.float32, copy=False))[None, ...],
    'mask': torch.from_numpy(mask_patch)[None, ...],
    'file_path_': [T0.strftime('SM_%Y%m%d%H%M')],
}


In [ ]:
dt = datetime.strptime(el['file_path_'][0], 'SM_%Y%m%d%H%M')
mask = el['mask'].cpu().numpy().squeeze()
data = rearrange(el['image'].cpu().numpy().squeeze(), 'h w s -> s h w')
input_data = np.ma.array(data, mask=np.broadcast_to(mask, data.shape))


# Plot input soil moisture sequence


In [ ]:
# Plot the input soil moisture sequence
# Note: `input_data` is defined in the previous cell that converts `el`.
assert 'input_data' in globals(), 'input_data is not defined. Run the cell above that builds input_data first.'
input_phys = swvl1_norm_to_phys(input_data)
context_titles = [f"D-{input_phys.shape[0] - idx - 1}" if idx < input_phys.shape[0] - 1 else 'D0' for idx in range(input_phys.shape[0])]
plot_era5land_grid(
    input_phys,
    title=f'Input sequence | start {dt:%Y-%m-%d}',
    frame_titles=context_titles,
    colorbar=True,
    ncols=input_phys.shape[0],
    figsize=(2.0 * input_phys.shape[0] + 1.0, 2.8),
)


## Forecast the next days (context = 7 days)

Note: GPTCast inference uses up to 7 context steps by default (trained with an 8-step temporal window = 7 past + 1 predicted).


In [ ]:
# Note: multi-step forecasts can be slow (token-by-token generation).
# Default to the actual 7-day target task. For a quick smoke test, set FORECAST_STEPS=1 manually.
FORECAST_STEPS = 7
TOP_K = 1
TEMPERATURE = 1.0
# Interactive Jupyter default: token-level progress (`2`).
# Headless execution can override this, e.g. `GPTCAST_NOTEBOOK_VERBOSITY=0`.
VERBOSITY = int(os.environ.get('GPTCAST_NOTEBOOK_VERBOSITY', '2'))

forecast_main = gpt_main.forecast_swvl1(input_data, steps=FORECAST_STEPS, normalized=True, temperature=TEMPERATURE, top_k=TOP_K, verbosity=VERBOSITY)
forecast_compare = gpt_compare.forecast_swvl1(input_data, steps=FORECAST_STEPS, normalized=True, temperature=TEMPERATURE, top_k=TOP_K, verbosity=VERBOSITY) if gpt_compare is not None else None


### plot the forecasts


In [ ]:
forecast_rows = [swvl1_norm_to_phys(forecast_main)]
forecast_titles = [f'Forecast ({gpt_ckpt.stem})']
if forecast_compare is not None:
    forecast_rows.append(swvl1_norm_to_phys(forecast_compare))
    forecast_titles.append(f'Forecast ({Path(gpt_compare_ckpt).stem})')

lead_titles = [f'D+{i+1}' for i in range(forecast_rows[0].shape[0])]
plot_era5land_panel(
    forecast_rows,
    row_titles=forecast_titles,
    frame_titles=lead_titles,
    title='GPTCast forecast comparison',
    colorbar=True,
    figsize=(2.6 * len(lead_titles) + 1.0, 2.4 * len(forecast_rows) + 0.5),
)


In [ ]:
# --- Quantitative sanity check (paper-style logic) ---
# This evaluates a single reproducible East China ROI sample (land-heavy) with: 7-day context + N-day target.
# For paper-level reporting, aggregate these metrics over many samples / dates.

CONTEXT_STEPS = 7
EVAL_STEPS = 7  # default to the full week target task; set to 1 manually for a quick smoke test
EVAL_SEQ_LEN = CONTEXT_STEPS + EVAL_STEPS

TOP_K = 1
TEMPERATURE = 1.0

assert 'BASE_DIR' in globals(), 'Run the ROI extraction cell above first (defines BASE_DIR, ROI_CENTER, etc.).'

import xarray as xr
from datetime import timedelta
import torch.nn.functional as F

T0_EVAL = T0

def open_year_ds(year: int) -> xr.Dataset:
    p = BASE_DIR / str(year) / NC_NAME
    assert p.exists(), f"Missing: {p}"
    return xr.open_dataset(p)

def sel_time_swvl1(ds: xr.Dataset, dt: datetime):
    t = np.datetime64(dt)
    try:
        return ds['swvl1'].sel(time=t)
    except Exception:
        return ds['swvl1'].sel(time=t, method='nearest')

frames = []
ds_y = open_year_ds(T0_EVAL.year)
for j in range(EVAL_SEQ_LEN):
    dt_j = T0_EVAL + timedelta(days=j)
    da = sel_time_swvl1(ds_y, dt_j)
    frames.append(da.values.astype(np.float32, copy=False))
frames = np.stack(frames, axis=0)

lat_vals = ds_y['latitude'].values
lon_vals = ds_y['longitude'].values
mask_native = np.isnan(frames[0])

cmin, cmax = float(CLIP[0]), float(CLIP[1])
frames = np.nan_to_num(frames, nan=cmin)
frames = np.clip(frames, cmin, cmax)

x = torch.from_numpy(frames).unsqueeze(1)
x = F.interpolate(x, size=(RESIZE, RESIZE), mode='bilinear', align_corners=False)
frames_r = x.squeeze(1).numpy()

m = torch.from_numpy(mask_native.astype(np.float32))[None, None, ...]
m = F.interpolate(m, size=(RESIZE, RESIZE), mode='nearest')
mask_r = m[0, 0].numpy().astype(bool)

frames_n = (frames_r - cmin) / (cmax - cmin + 1e-12)
frames_n = frames_n * 2 - 1

lat_c = float(ROI_CENTER['lat'])
lon_c = float(ROI_CENTER['lon'])
lat_idx = int(np.argmin(np.abs(lat_vals - lat_c)))
lon_idx = int(np.argmin(np.abs(lon_vals - lon_c)))
y_center = int(round(lat_idx / (len(lat_vals) - 1) * (RESIZE - 1)))
x_center = int(round(lon_idx / (len(lon_vals) - 1) * (RESIZE - 1)))

def clamp(v, lo, hi):
    return int(max(lo, min(hi, v)))

best = None
best_frac = 1.0
rng_eval = random.Random(42)
for k in range(SMART_ATTEMPTS):
    dy = rng_eval.randint(-JITTER_PX, JITTER_PX) if k > 0 else 0
    dx = rng_eval.randint(-JITTER_PX, JITTER_PX) if k > 0 else 0
    y0 = clamp(y_center - CROP // 2 + dy, 0, RESIZE - CROP)
    x0 = clamp(x_center - CROP // 2 + dx, 0, RESIZE - CROP)
    frac = float(mask_r[y0 : y0 + CROP, x0 : x0 + CROP].mean())
    if frac < best_frac:
        best_frac = frac
        best = (y0, x0)
    if frac <= MAX_MASK_FRAC:
        best = (y0, x0)
        best_frac = frac
        break

y0, x0 = best
print(f"Eval ROI crop mask_frac={best_frac:.3f}  start={T0_EVAL}  context={CONTEXT_STEPS}  steps={EVAL_STEPS}")

patch = frames_n[:, y0 : y0 + CROP, x0 : x0 + CROP]
mask_patch = mask_r[y0 : y0 + CROP, x0 : x0 + CROP]

seq_eval = np.ma.array(patch, mask=np.broadcast_to(mask_patch, patch.shape))
dt_eval = T0_EVAL
context = seq_eval[:CONTEXT_STEPS]
target = seq_eval[CONTEXT_STEPS:CONTEXT_STEPS + EVAL_STEPS]

def per_lead_mae_rmse(pred: np.ma.MaskedArray, tgt: np.ma.MaskedArray):
    diff = pred - tgt
    mae = np.ma.mean(np.abs(diff), axis=(1, 2))
    rmse = np.sqrt(np.ma.mean(diff ** 2, axis=(1, 2)))
    return mae, rmse

persist = np.ma.array(
    np.broadcast_to(context[-1][None, ...], target.shape),
    mask=np.broadcast_to(mask_patch, target.shape),
)

pred_main = gpt_main.forecast_swvl1(context, steps=EVAL_STEPS, normalized=True, temperature=TEMPERATURE, top_k=TOP_K, verbosity=1)
pred_compare = gpt_compare.forecast_swvl1(context, steps=EVAL_STEPS, normalized=True, temperature=TEMPERATURE, top_k=TOP_K, verbosity=1) if gpt_compare is not None else None

tgt_phys = swvl1_norm_to_phys(target)
pers_phys = swvl1_norm_to_phys(persist)
main_phys = swvl1_norm_to_phys(pred_main)

mae_p, rmse_p = per_lead_mae_rmse(pers_phys, tgt_phys)
mae_m, rmse_m = per_lead_mae_rmse(main_phys, tgt_phys)

print(f"Eval sample start: {dt_eval}  |  context={CONTEXT_STEPS}  steps={EVAL_STEPS}  top_k={TOP_K}")
if pred_compare is None:
    print('lead  |  persistence_rmse  model_rmse')
    for i in range(EVAL_STEPS):
        print(f"+{i+1:02d}   |  {rmse_p[i]:.4f}            {rmse_m[i]:.4f}")
else:
    compare_phys = swvl1_norm_to_phys(pred_compare)
    mae_c, rmse_c = per_lead_mae_rmse(compare_phys, tgt_phys)
    print('lead  |  persistence_rmse  main_rmse  compare_rmse')
    for i in range(EVAL_STEPS):
        print(f"+{i+1:02d}   |  {rmse_p[i]:.4f}            {rmse_m[i]:.4f}       {rmse_c[i]:.4f}")

panel_rows = [tgt_phys, pers_phys, main_phys]
row_titles = ['Ground truth', 'Persistence', f'GPTCast ({gpt_ckpt.stem})']
if pred_compare is not None:
    panel_rows.append(compare_phys)
    row_titles.append(f'GPTCast ({Path(gpt_compare_ckpt).stem})')

lead_titles = [f'D+{i+1}' for i in range(EVAL_STEPS)]
plot_era5land_panel(
    panel_rows,
    row_titles=row_titles,
    frame_titles=lead_titles,
    title=f'One-sample sanity check | start {dt_eval:%Y-%m-%d}',
    colorbar=True,
    figsize=(2.8 * EVAL_STEPS + 1.1, 2.4 * len(panel_rows) + 0.5),
)
